# 01 Data Exploration — Verifying Data Availability

## Purpose
Before building any model, confirm that every data source is accessible
and understand the coverage, frequency, and missing-value profile of each series.
This notebook is the single source of truth for "what data do we actually have."

## Questions this notebook answers
- Which tickers return data reliably via yfinance / stooq?
- What date range and number of trading days are available per source?
- Are there significant gaps (holidays, suspensions, API failures)?
- Do all series share enough overlapping dates to be merged into one feature matrix?

## Expected outputs
| Output | Location |
|--------|----------|
| `close_all.csv` — merged Close prices for all tickers | `data/raw/` |
| Data-availability summary table (rows × missing %) | displayed in notebook |

## Sections
| # | Content |
|---|---------|
| 0 | Colab environment setup |
| 1 | Nikkei 225 (target) |
| 2 | Macro: equity indices & futures |
| 3 | Macro: FX rates |
| 4 | Macro: commodities |
| 5 | Macro: US Treasury yields |
| 6 | Macro: JGB 10Y yield (stooq) |
| 7 | Micro: top Nikkei 225 constituents |
| 8 | Summary & missing-value overview |
| 9 | Save raw data |
| 10 | Verify fetcher.py |


## 0. Environment Setup (Google Colab)

Run this cell only when executing on Google Colab.

In [ ]:
# ── Step 1: Install packages ───────────────────────────────────────────────
import subprocess, sys

pkgs = ['yfinance', 'pandas-ta', 'scikit-learn>=1.5.0', 'statsmodels>=0.14.0']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
out = result.stdout + result.stderr
print(out.strip() if out.strip() else 'All packages already up to date.')

if 'Successfully installed' in out:
    print('\nPackages upgraded — restarting runtime...')
    import os; os.kill(os.getpid(), 9)

In [ ]:
# ── Step 2: Repository & path setup ───────────────────────────────────────
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull

    os.chdir(REPO)

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    if REPO not in sys.path:
        sys.path.insert(0, REPO)

    print(f'Setup complete.  CWD={os.getcwd()}  path[0]={sys.path[0]}')
else:
    print('Running in local environment.')

In [ ]:
import os, sys, warnings

# ---------------------------------------------------------------------------
# Path setup — works whether Cell 0 was run or not
# ---------------------------------------------------------------------------
REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO):
        print('Repo not found — cloning...')
        os.system(f'git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}')
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.4f}'.format)

PERIOD   = '2y'
INTERVAL = '1d'
DATA_RAW = 'data/raw'

END   = datetime.today()
START = END - timedelta(days=365 * 2)
print(f'Ready. CWD={os.getcwd()}')

## 1. Target: Nikkei 225

In [ ]:
nikkei = yf.download('^N225', period=PERIOD, interval=INTERVAL, progress=False)
print(f'Rows   : {len(nikkei)}')
print(f'Period : {nikkei.index[0].date()} to {nikkei.index[-1].date()}')
print(f'Missing: {nikkei.isnull().sum().to_dict()}')
display(nikkei.tail())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
nikkei['Close'].plot(ax=axes[0], title='Nikkei 225 Close')
nikkei['Volume'].plot(ax=axes[1], title='Volume', color='orange')
axes[0].set_ylabel('Price (JPY)')
axes[1].set_ylabel('Volume')
plt.tight_layout()
plt.show()

## 2. Macro: Futures & Equity Indices

In [ ]:
macro_index_tickers = {
    'Nikkei225_Futures': 'NKD=F',
    'SP500':             '^GSPC',
    'NASDAQ':            '^IXIC',
    'DowJones':          '^DJI',
    'VIX':               '^VIX',
}

results_index = {}
for name, ticker in macro_index_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
    results_index[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<25} ticker={ticker:<10} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_index.items():
    if len(df) > 0:
        (df['Close'] / df['Close'].iloc[0] * 100).plot(ax=ax, label=name)
(nikkei['Close'] / nikkei['Close'].iloc[0] * 100).plot(
    ax=ax, label='Nikkei225', linewidth=2, linestyle='--', color='black')
ax.set_title('Equity Indices — Normalized (base=100)')
ax.set_ylabel('Index (base=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Macro: FX Rates

In [ ]:
fx_tickers = {
    'USD_JPY': 'JPY=X',
    'EUR_JPY': 'EURJPY=X',
    'GBP_JPY': 'GBPJPY=X',
    'CNY_JPY': 'CNYJPY=X',
}

results_fx = {}
for name, ticker in fx_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
    results_fx[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<12} ticker={ticker:<12} rows={len(df):4d}  close_na={na}")

In [ ]:
n = len(results_fx)
fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
for ax, (name, df) in zip(axes, results_fx.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
        ax.set_ylabel('Rate (JPY)')
plt.tight_layout()
plt.show()

## 4. Macro: Commodities

In [ ]:
commodity_tickers = {
    'WTI_Crude': 'CL=F',
    'Gold':      'GC=F',
    'Silver':    'SI=F',
    'Copper':    'HG=F',
}

results_commodity = {}
for name, ticker in commodity_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
    results_commodity[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<12} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
n = len(results_commodity)
fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
for ax, (name, df) in zip(axes, results_commodity.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
        ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

## 5. Macro: US Treasury Yields

In [ ]:
bond_tickers = {
    'US_2Y':  '^IRX',
    'US_10Y': '^TNX',
    'US_30Y': '^TYX',
}

results_bond = {}
for name, ticker in bond_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
    results_bond[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<8} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_bond.items():
    if len(df) > 0:
        df['Close'].plot(ax=ax, label=name)
ax.set_title('US Treasury Yields')
ax.set_ylabel('Yield (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Macro: JGB 10Y Yield (stooq)

In [ ]:
try:
    start_str = START.strftime('%Y%m%d')
    end_str   = END.strftime('%Y%m%d')
    url = f'https://stooq.com/q/d/l/?s=10lj.b&d1={start_str}&d2={end_str}&i=d'
    df_jgb = pd.read_csv(url, parse_dates=['Date'], index_col='Date').sort_index()
    na = int(df_jgb['Close'].isnull().sum())
    print(f'OK | stooq  rows={len(df_jgb)}  close_na={na}')
    display(df_jgb.tail())
except Exception as e:
    df_jgb = pd.DataFrame()
    print(f'NG | stooq  error: {e}')

In [ ]:
if len(df_jgb) > 0:
    fig, ax = plt.subplots(figsize=(12, 3))
    df_jgb['Close'].plot(ax=ax, title='JGB 10Y Yield (stooq)', color='steelblue')
    ax.set_ylabel('Yield (%)')
    plt.tight_layout()
    plt.show()

## 7. Micro: Nikkei 225 Major Constituents

In [ ]:
micro_tickers = {
    'FastRetailing':  '9983.T',
    'SoftBank':       '9984.T',
    'Tokyo_Electron': '8035.T',
    'KDDI':           '9433.T',
    'ShinEtsu_Chem':  '4063.T',
}

results_micro = {}
for name, ticker in micro_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, progress=False)
    results_micro[name] = df
    ok = len(df) > 0
    na = int(df['Close'].isnull().sum()) if ok else '-'
    print(f"{'OK' if ok else 'NG'} | {name:<20} ticker={ticker:<8} rows={len(df):4d}  close_na={na}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_micro.items():
    if len(df) > 0:
        (df['Close'] / df['Close'].iloc[0] * 100).plot(ax=ax, label=name)
ax.set_title('Nikkei 225 Major Constituents — Normalized (base=100)')
ax.set_ylabel('Index (base=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
all_results = {
    'Nikkei225':                 nikkei,
    **{f'[Index] {k}':      v for k, v in results_index.items()},
    **{f'[FX] {k}':         v for k, v in results_fx.items()},
    **{f'[Commodity] {k}':  v for k, v in results_commodity.items()},
    **{f'[Bond_US] {k}':    v for k, v in results_bond.items()},
    '[JGB] JGB_10Y':             df_jgb if len(df_jgb) > 0 else pd.DataFrame(),
    **{f'[Micro] {k}':      v for k, v in results_micro.items()},
}

summary = []
for name, df in all_results.items():
    if len(df) > 0 and 'Close' in df.columns:
        summary.append({
            'series':   name,
            'ok':       True,
            'rows':     len(df),
            'start':    df.index[0].date(),
            'end':      df.index[-1].date(),
            'close_na': int(df['Close'].isnull().sum()),
            'na_pct':   round(df['Close'].isnull().mean() * 100, 1),
        })
    else:
        summary.append({'series': name, 'ok': False,
                        'rows': 0, 'start': None, 'end': None,
                        'close_na': None, 'na_pct': None})

display(pd.DataFrame(summary))

In [ ]:
# Merge all Close prices into one DataFrame
close_df = pd.DataFrame({'Nikkei225': nikkei['Close'].squeeze()})

for name, df in {**results_index, **results_fx, **results_commodity, **results_bond, **results_micro}.items():
    if len(df) > 0 and 'Close' in df.columns:
        close_df[name] = df['Close'].squeeze()

if len(df_jgb) > 0:
    close_df['JGB_10Y'] = df_jgb['Close']

print(f'Shape : {close_df.shape}')
print('\nMissing rate (%):')
print((close_df.isnull().mean() * 100).round(1).to_string())
display(close_df.tail())

## 9. Save Raw Data

In [ ]:
os.makedirs(DATA_RAW, exist_ok=True)
save_path = f'{DATA_RAW}/close_all.csv'
close_df.to_csv(save_path)
print(f'Saved: {save_path}  shape={close_df.shape}')

## 10. Verify fetcher.py

Confirm that `src/data/fetcher.py` produces the same result as the manual fetch above.

In [ ]:
from src.data.fetcher import fetch_all

df_all = fetch_all(period='2y', interval='1d')
print(f'Shape : {df_all.shape}')
print('\nColumns:', df_all.columns.tolist())
display(df_all.tail())

## 11. Next Steps

- `src/data/fetcher.py` is ready — use `fetch_all()` in subsequent notebooks
- Review missing rates and decide on imputation strategy (forward-fill recommended for trading-day gaps)
- Next notebook: `02_correlation.ipynb` — correlation analysis and multiple regression